In [1]:
import os
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, IntSlider

ADNI_BASE_PATH = os.getenv("ADNI_BASE_PATH")
ADNI_BASE_PATH

'/Volumes/Corsair/agpmd/ADNI_flat'

In [8]:
# subject_id = "012_S_4026"
subject_id = "067_S_6045"

In [9]:

# 1. Define the exact file paths
mri_path = f'{ADNI_BASE_PATH}/{subject_id}/mri.nii'
pet_path = f'{ADNI_BASE_PATH}/{subject_id}/pet.nii.gz'

# 2. Load the NIfTI files into memory
print("Loading NIfTI files...")
mri_img = nib.load(mri_path)
pet_img = nib.load(pet_path)

# Extract the raw 3D NumPy arrays
mri_data = mri_img.get_fdata()
pet_data = pet_img.get_fdata()

# Handle potential 4D data (ADNI sometimes leaves a time dimension of size 1, e.g., 160x192x160x1)
if mri_data.ndim > 3: mri_data = np.squeeze(mri_data)
if pet_data.ndim > 3: pet_data = np.squeeze(pet_data)

print(f"MRI Shape: {mri_data.shape}")
print(f"PET Shape: {pet_data.shape}")

# 3. Create the interactive visualization function
def explore_3d_images(mri_z, pet_z):
    """
    Plots a 2D slice of the 3D volume.
    We use .T (transpose) and origin='lower' to orient the brain upright.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Plot MRI ---
    # Grayscale is standard for structural MRI
    axes[0].imshow(mri_data[:, :, mri_z].T, cmap='gray', origin='lower')
    axes[0].set_title(f'MRI (MT1 N3m) - Axial Slice {mri_z}')
    axes[0].axis('off')

    # --- Plot PET ---
    # 'hot' or 'jet' colormaps are standard for highlighting PET tracer uptake
    axes[1].imshow(pet_data[:, :, pet_z].T, cmap='hot', origin='lower')
    axes[1].set_title(f'PET (AV45 Amyloid) - Axial Slice {pet_z}')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# 4. Generate Interactive Sliders
# We create separate sliders because the MRI and PET likely have different
# native dimensions until you run your co-registration pipeline.
mri_max = mri_data.shape[2] - 1
pet_max = pet_data.shape[2] - 1

print("\nUse the sliders below to scroll through the brain volumes:")
interact(
    explore_3d_images,
    mri_z=IntSlider(min=0, max=mri_max, step=1, value=mri_max//2, description='MRI Slice'),
    pet_z=IntSlider(min=0, max=pet_max, step=1, value=pet_max//2, description='PET Slice')
);

Loading NIfTI files...
MRI Shape: (208, 240, 256)
PET Shape: (160, 160, 96)

Use the sliders below to scroll through the brain volumes:


interactive(children=(IntSlider(value=127, description='MRI Slice', max=255), IntSlider(value=47, description=…